# Notebook 2 — Modeling
Supervised models + anomaly detection with full evaluation.
Covers: SMOTE, XGBoost, Isolation Forest, PR curves, ROC curves.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys, os
sys.path.append(os.path.join('..', 'src'))
from fraud_detector import FraudDetectionPipeline, generate_synthetic_fraud_data

# Run full pipeline
pipeline = FraudDetectionPipeline(output_dir='../reports')
df = generate_synthetic_fraud_data()
pipeline.load_and_explore(df)
pipeline.preprocess()
X_res, y_res = pipeline.apply_resampling(strategy='smote')
pipeline.train_supervised_models(X_res, y_res)
pipeline.train_anomaly_detectors()
pipeline.get_feature_importance()
print('Pipeline done.')

## Precision-Recall Curves
PR-AUC is the primary metric — ROC-AUC is inflated by the huge true-negative count.

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#1D9E75', '#378ADD', '#E24B4A', '#BA7517', '#7F77DD']

for (name, metrics), color in zip(pipeline.results.items(), colors):
    y_prob = np.array(metrics['y_prob'])
    prec, rec, _ = precision_recall_curve(pipeline.y_test, y_prob)
    ap = metrics['pr_auc']
    ax.plot(rec, prec, label=f'{name} (PR-AUC={ap:.3f})', color=color, linewidth=2)

fraud_rate = pipeline.y_test.mean()
ax.axhline(fraud_rate, color='gray', linestyle='--', linewidth=1,
           label=f'Random baseline ({fraud_rate:.4f})')

ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curves', fontsize=13, fontweight='bold')
ax.legend(loc='upper right', fontsize=9)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.05])
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../reports/figures/pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## ROC Curves

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#1D9E75', '#378ADD', '#E24B4A', '#BA7517', '#7F77DD']

for (name, metrics), color in zip(pipeline.results.items(), colors):
    y_prob = np.array(metrics['y_prob'])
    fpr, tpr, _ = roc_curve(pipeline.y_test, y_prob)
    auc = metrics['roc_auc']
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=color, linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves\n(Note: inflated by large TN count — use PR-AUC as primary metric)',
             fontsize=11, fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../reports/figures/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## Confusion Matrices

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

n = len(pipeline.results)
fig, axes = plt.subplots(1, n, figsize=(4*n, 4))

for ax, (name, metrics) in zip(axes, pipeline.results.items()):
    cm = np.array([[metrics['tn'], metrics['fp']],
                   [metrics['fn'], metrics['tp']]])
    sns.heatmap(cm, annot=True, fmt='d', ax=ax, cmap='Blues',
                xticklabels=['Pred Legit', 'Pred Fraud'],
                yticklabels=['Actual Legit', 'Actual Fraud'],
                cbar=False)
    ax.set_title(f'{name}\nthresh={metrics["threshold"]:.3f}', fontsize=9, fontweight='bold')

plt.suptitle('Confusion Matrices (at business-cost-optimal threshold)', y=1.02, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## Feature Importance

In [ ]:
features, importances = zip(*pipeline.feature_importances)
colors = ['#E24B4A' if imp > 0.4 else '#BA7517' if imp > 0.1 else '#378ADD'
          for imp in importances]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(features[::-1], [i*100 for i in importances[::-1]], color=colors[::-1])
ax.set_xlabel('Importance (%)')
ax.set_title('Top 15 Feature Importances (avg across tree models)', fontweight='bold')
for bar, val in zip(bars, [i*100 for i in importances[::-1]]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=9)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../reports/figures/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## Model Comparison Summary

In [ ]:
summary = pd.DataFrame([
    {
        'Model': name,
        'PR-AUC': f"{m['pr_auc']:.4f}",
        'ROC-AUC': f"{m['roc_auc']:.4f}",
        'Precision': f"{m['precision']:.4f}",
        'Recall': f"{m['recall']:.4f}",
        'F1': f"{m['f1']:.4f}",
        'Threshold': f"{m['threshold']:.3f}",
    }
    for name, m in pipeline.results.items()
])
summary = summary.sort_values('PR-AUC', ascending=False).reset_index(drop=True)
print(summary.to_string(index=False))